In [ ]:
%pip install qiskit qiskit-aer matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

In [ ]:
n = 5
qc = QuantumCircuit(n)

# Prepare an example input |00101> (decimal 5)
# Qiskit uses qubit 0 as the least-significant bit.
input_value = 5
for qubit in range(n):
    if (input_value >> qubit) & 1:
        qc.x(qubit)

qc.barrier()

# Apply the Quantum Fourier Transform
for target in range(n):
    qc.h(target)
    for control in range(target + 1, n):
        angle = np.pi / (2 ** (control - target))
        qc.cp(angle, control, target)

# Reverse qubit order to match the standard QFT output ordering
for qubit in range(n // 2):
    qc.swap(qubit, n - qubit - 1)

qc.draw(output="mpl")

In [ ]:
state = Statevector.from_instruction(qc)
probabilities = state.probabilities()

labels = [format(i, "05b") for i in range(2**n)]

plt.figure(figsize=(12, 4))
plt.bar(labels, probabilities)
plt.xticks(rotation=90)
plt.xlabel("Computational basis state")
plt.ylabel("Probability")
plt.title("5-Qubit QFT Output Probabilities")
plt.tight_layout()
plt.show()

print("Number of qubits:", n)
print("Number of basis states:", 2**n)
print("Sum of probabilities:", round(float(np.sum(probabilities)), 6))

In [ ]:
# Qiskit provides the standard QFT circuit through its library.
# Build an inverse-QFT circuit and apply it to the output.
inverse_qft = qc.copy()
# Remove the input-state preparation and keep only the QFT operations
# Rebuild the QFT-only circuit for a clear verification.
qft = QuantumCircuit(n)

for target in range(n):
    qft.h(target)
    for control in range(target + 1, n):
        angle = np.pi / (2 ** (control - target))
        qft.cp(angle, control, target)

for qubit in range(n // 2):
    qft.swap(qubit, n - qubit - 1)

test_circuit = QuantumCircuit(n)
for qubit in range(n):
    if (input_value >> qubit) & 1:
        test_circuit.x(qubit)

test_circuit.compose(qft, inplace=True)
test_circuit.compose(qft.inverse(), inplace=True)

recovered_state = Statevector.from_instruction(test_circuit)
recovered_probabilities = recovered_state.probabilities()

print("Probability of recovering the original state |00101>:", 
      round(float(recovered_probabilities[input_value]), 6))

## Conclusion
The experiment demonstrates the QFT using Hadamard gates, controlled phase-rotation gates, and SWAP gates. The inverse-QFT check confirms that the original input state can be recovered.